# Benchmark - run a two-player grid on Colab
Plays the two-player games of a grid (`grids/duel_*.yaml`: chess, Connect Four) that have not been played yet, one model at a time, and saves one record per game on Google Drive, next to the records of the textual games.
The agent is the same of the textual games (same modes), with the prompts of `duel_prompts.py`; the levels of a game are its opponents (`random`, `greedy`, Stockfish `sf<N>` / `elo<N>`, or the model itself `self-<mode>`, see `duel.py`).
If the run stops (or the runtime is deleted), run the notebook again: it resumes from the games not yet saved.

Recommended order: a pilot (`limit = 8`) to see the speed and the context sizes, then `duel_tier1.yaml`, then `duel_selfplay.yaml`.

# Setup
Clones the repository (only the benchmark folder: the two-player games need no game files), installs vLLM, python-chess and Stockfish, and moves into `Code/Benchmark`. Locally it does nothing.
The first time on a new Colab machine the installation takes a few minutes: restart the kernel after it (and then run all the cells), because it can update torch.

In [ ]:
import os
import sys
import shutil
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/UPROOT01/LLMs-Play-Textual-Games.git"
    repo_dir = "/content/LLMs-Play-Textual-Games"
    if not os.path.isfile(f"{repo_dir}/Code/Benchmark/run.py"):
        shutil.rmtree(repo_dir, ignore_errors=True) # leftover of a failed clone, or a clone without the benchmark
        # sparse clone: only the folders needed (the whole Games folder is >1 GB)
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", repo_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", repo_dir, "sparse-checkout", "add", "Code/Benchmark"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.31.0", "pyyaml", "chess"], check=True)
    subprocess.run(["apt-get", "install", "-qq", "-y", "stockfish"], check=True) # the sf<N> / elo<N> opponents
    os.chdir(f"{repo_dir}/Code/Benchmark")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU available: connect to a Colab runtime with a GPU (e.g. L4)"
! nvidia-smi

# What to run
- `grid`: the grid file (`grids/duel_*.yaml`). `models`: only these models of the grid (`None`: all of them, one after the other).
- `limit`: at most this many games per model (`None`: all). Use it for a pilot.
- `n_parallel`: games played at the same time. vLLM answers them together: more games, more tokens per minute, until the GPU memory is full.
  `"auto"` decides by itself: it starts a new game only if the contexts of all the games, at the size they are expected to reach, fit in the GPU memory (see `engine.ContextBudget`).
  The status line shows how full the GPU memory is expected to be. A number (e.g. `8`) plays always that many.

To change the models or their vLLM settings (e.g. a smaller `max_model_len` if a model does not fit), see `profiles.py`.

In [ ]:
from records import LEGACY_RESULTS
from run import summary
from run_duel import load_duel_grid, run_duel_grid

grid = load_duel_grid("grids/duel_tier1.yaml")
models = ["Qwen/Qwen3-1.7B"] # None: all the models of the grid
# models = None
limit = 8 # e.g. 8 for a pilot
n_parallel = "auto"

if IN_COLAB:
    assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not mounted: run the cell above"
    results = "/content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results"
else:
    results = "results_new"
read = [LEGACY_RESULTS] # the old experiments: their games are not played again
print("Results folder:", results)
summary(grid, results, read, models)

# Run
Every game is saved as soon as it ends. Stopping the cell is safe: the games being played are lost, the others are kept.
If loading the second model of a run fails because the GPU memory is full, restart the runtime and run everything again: the first model's games are skipped.

In [ ]:
run_duel_grid(grid, results, read, models=models, limit=limit, n_parallel=n_parallel)

# Quick look
Per model, game, opponent and mode: average score (win 1, draw 0.5, loss 0), wins, draws and losses, illegal answers and random moves played for the agent per game, thousands of generated tokens per game.

In [ ]:
from records import load_table

table = load_table(results)
games = table[(table.source == "benchmark-duel") & table.game.isin(grid["games"]) & table.model.isin(grid["models"])
              & table["mode"].isin(grid["modes"]) & table.seed.isin(grid["seeds"])]
games.groupby(["model", "game", "opponent", "mode"]).agg(
    score=("score", "mean"), wins=("result", lambda r: (r == "win").sum()), draws=("result", lambda r: (r == "draw").sum()),
    losses=("result", lambda r: (r == "loss").sum()), illegal=("illegal", "mean"), forced=("forced", "mean"),
    ktokens=("tokens_generated", lambda t: t.mean() / 1000)).round(2)